In [ ]:
import arcpy
import os

# =========================
# 1) 环境与输入
# =========================
arcpy.env.overwriteOutput = True
scratch_gdb = arcpy.env.scratchGDB

china_province = r"D:\path\to\sinkhole-risk-china\data\Administrative_divisions_of_china\china_pro2.shp"

out_grid_points_wgs84 = os.path.join(scratch_gdb, "China_20km_grid_points_WGS84")

cell_size = 20000  # 20 km（米）

print("读取省界数据...")
desc = arcpy.Describe(china_province)
sr_in = desc.spatialReference

# ✅ 你的省界字段：ADCODE99 + NAME
need_fields = ["ADCODE99", "NAME"]
have_fields = [f.name for f in arcpy.ListFields(china_province)]
missing = [f for f in need_fields if f not in have_fields]
if missing:
    raise RuntimeError(f"省界数据缺少字段：{missing}。请检查 {china_province} 的字段名是否一致。")

# =========================
# 2) 确保用于生成网格的坐标系单位是“米”
# =========================
def is_degree_sr(spref: arcpy.SpatialReference) -> bool:
    try:
        return (spref.type == "Geographic") or (spref.linearUnitName is None) or ("Degree" in (spref.linearUnitName or ""))
    except:
        return True

# 建议投影：Asia North Albers Equal Area Conic（单位米）
try:
    sr_grid = arcpy.SpatialReference("Asia North Albers Equal Area Conic")
except:
    sr_grid = arcpy.SpatialReference(3857)  # 兜底（单位米）

china_for_grid = china_province
if is_degree_sr(sr_in):
    print("检测到省界为地理坐标系（度）。正在投影到米单位投影坐标系以生成20km网格...")
    china_for_grid = os.path.join(scratch_gdb, "china_province_projected_for_grid")
    arcpy.management.Project(china_province, china_for_grid, sr_grid)
else:
    sr_grid = sr_in

china_extent = arcpy.Describe(china_for_grid).extent
arcpy.env.outputCoordinateSystem = sr_grid

# =========================
# 3) 创建20km渔网（POLYGON）
# =========================
print("创建20km渔网（POLYGON）...")
fishnet = os.path.join(scratch_gdb, "fishnet_20km_poly")

origin_coord = f"{china_extent.XMin} {china_extent.YMin}"
y_axis_coord = f"{china_extent.XMin} {china_extent.YMax}"
corner_coord = f"{china_extent.XMax} {china_extent.YMax}"

arcpy.management.CreateFishnet(
    out_feature_class=fishnet,
    origin_coord=origin_coord,
    y_axis_coord=y_axis_coord,
    cell_width=cell_size,
    cell_height=cell_size,
    number_rows=0,
    number_columns=0,
    corner_coord=corner_coord,
    labels="NO_LABELS",
    template="#",
    geometry_type="POLYGON"
)

# =========================
# 4) 渔网转中心点（质心）
# =========================
print("生成渔网中心点（质心）...")
grid_points = os.path.join(scratch_gdb, "fishnet_20km_centroids")
arcpy.management.FeatureToPoint(fishnet, grid_points, "CENTROID")

# =========================
# 5) 裁剪到中国范围
# =========================
print("裁剪到中国范围...")
clipped_points = os.path.join(scratch_gdb, "China_20km_grid_points_projected")
arcpy.analysis.Clip(grid_points, china_for_grid, clipped_points)

# =========================
# 6) Spatial Join：带出 ADCODE99 + NAME
# =========================
print("为网格点添加省份属性（ADCODE99 / NAME）...")
points_joined = os.path.join(scratch_gdb, "China_20km_grid_points_joined")

arcpy.analysis.SpatialJoin(
    target_features=clipped_points,
    join_features=china_for_grid,
    out_feature_class=points_joined,
    join_operation="JOIN_ONE_TO_ONE",
    join_type="KEEP_COMMON",
    match_option="INTERSECT"
)

# =========================
# 7) 投影到WGS84
# =========================
print("投影到WGS84...")
wgs84 = arcpy.SpatialReference(4326)
arcpy.management.Project(points_joined, out_grid_points_wgs84, wgs84)

# =========================
# 8) 添加 Province 字段并用 NAME 填充
# =========================
print("添加 Province 字段并用 NAME 填充...")
fields_out = [f.name for f in arcpy.ListFields(out_grid_points_wgs84)]
if "Province" not in fields_out:
    arcpy.management.AddField(out_grid_points_wgs84, "Province", "TEXT", field_length=100)

# 计算 Province = NAME
# 注：如果 join 后 NAME 变成 NAME_1，这里做个兼容处理
fields_out = [f.name for f in arcpy.ListFields(out_grid_points_wgs84)]
name_field = "NAME" if "NAME" in fields_out else ("NAME_1" if "NAME_1" in fields_out else None)
if name_field is None:
    raise RuntimeError("输出结果中找不到 NAME 或 NAME_1 字段，请检查 Spatial Join 输出字段。")

with arcpy.da.UpdateCursor(out_grid_points_wgs84, [name_field, "Province"]) as cursor:
    for n, p in cursor:
        cursor.updateRow((n, n))

# =========================
# 9) 添加经纬度字段
# =========================
print("添加经纬度字段...")
fields_out = [f.name for f in arcpy.ListFields(out_grid_points_wgs84)]
if "Longitude" not in fields_out:
    arcpy.management.AddField(out_grid_points_wgs84, "Longitude", "DOUBLE")
if "Latitude" not in fields_out:
    arcpy.management.AddField(out_grid_points_wgs84, "Latitude", "DOUBLE")

with arcpy.da.UpdateCursor(out_grid_points_wgs84, ["SHAPE@X", "SHAPE@Y", "Longitude", "Latitude"]) as cursor:
    for x, y, lon, lat in cursor:
        cursor.updateRow((x, y, x, y))

# =========================
# 10) 加载到当前地图（可选）
# =========================
print("添加结果到当前地图...")
project = arcpy.mp.ArcGISProject("CURRENT")
active_map = project.activeMap
active_map.addDataFromPath(out_grid_points_wgs84)

count = int(arcpy.management.GetCount(out_grid_points_wgs84)[0])
print(f"完成！中国范围内生成20km均匀网格点：{count} 个")
print("字段包含：Longitude, Latitude, ADCODE99, Province（由NAME复制而来）")
print(f"输出要素：{out_grid_points_wgs84}")


In [ ]:
import arcpy
import os

# =========================
# 1) 环境与输入
# =========================
arcpy.env.overwriteOutput = True
scratch_gdb = arcpy.env.scratchGDB

china_province = r"D:\path\to\sinkhole-risk-china\data\Administrative_divisions_of_china\china_pro2.shp"

out_grid_points_wgs84 = os.path.join(scratch_gdb, "China_10km_grid_points_WGS84")

cell_size = 10000  # 10 km（米）

print("读取省界数据...")
desc = arcpy.Describe(china_province)
sr_in = desc.spatialReference

# ✅ 你的省界字段：ADCODE99 + NAME
need_fields = ["ADCODE99", "NAME"]
have_fields = [f.name for f in arcpy.ListFields(china_province)]
missing = [f for f in need_fields if f not in have_fields]
if missing:
    raise RuntimeError(f"省界数据缺少字段：{missing}。请检查 {china_province} 的字段名是否一致。")

# =========================
# 2) 确保用于生成网格的坐标系单位是“米”
# =========================
def is_degree_sr(spref: arcpy.SpatialReference) -> bool:
    try:
        return (spref.type == "Geographic") or (spref.linearUnitName is None) or ("Degree" in (spref.linearUnitName or ""))
    except:
        return True

# 建议投影：Asia North Albers Equal Area Conic（单位米）
try:
    sr_grid = arcpy.SpatialReference("Asia North Albers Equal Area Conic")
except:
    sr_grid = arcpy.SpatialReference(3857)  # 兜底（单位米）

china_for_grid = china_province
if is_degree_sr(sr_in):
    print("检测到省界为地理坐标系（度）。正在投影到米单位投影坐标系以生成10km网格...")
    china_for_grid = os.path.join(scratch_gdb, "china_province_projected_for_grid")
    arcpy.management.Project(china_province, china_for_grid, sr_grid)
else:
    sr_grid = sr_in

china_extent = arcpy.Describe(china_for_grid).extent
arcpy.env.outputCoordinateSystem = sr_grid

# =========================
# 3) 创建10km渔网（POLYGON）
# =========================
print("创建10km渔网（POLYGON）...")
fishnet = os.path.join(scratch_gdb, "fishnet_10km_poly")

origin_coord = f"{china_extent.XMin} {china_extent.YMin}"
y_axis_coord = f"{china_extent.XMin} {china_extent.YMax}"
corner_coord = f"{china_extent.XMax} {china_extent.YMax}"

arcpy.management.CreateFishnet(
    out_feature_class=fishnet,
    origin_coord=origin_coord,
    y_axis_coord=y_axis_coord,
    cell_width=cell_size,
    cell_height=cell_size,
    number_rows=0,
    number_columns=0,
    corner_coord=corner_coord,
    labels="NO_LABELS",
    template="#",
    geometry_type="POLYGON"
)

# =========================
# 4) 渔网转中心点（质心）
# =========================
print("生成渔网中心点（质心）...")
grid_points = os.path.join(scratch_gdb, "fishnet_10km_centroids")
arcpy.management.FeatureToPoint(fishnet, grid_points, "CENTROID")

# =========================
# 5) 裁剪到中国范围
# =========================
print("裁剪到中国范围...")
clipped_points = os.path.join(scratch_gdb, "China_10km_grid_points_projected")
arcpy.analysis.Clip(grid_points, china_for_grid, clipped_points)

# =========================
# 6) Spatial Join：带出 ADCODE99 + NAME
# =========================
print("为网格点添加省份属性（ADCODE99 / NAME）...")
points_joined = os.path.join(scratch_gdb, "China_10km_grid_points_joined")

arcpy.analysis.SpatialJoin(
    target_features=clipped_points,
    join_features=china_for_grid,
    out_feature_class=points_joined,
    join_operation="JOIN_ONE_TO_ONE",
    join_type="KEEP_COMMON",
    match_option="INTERSECT"
)

# =========================
# 7) 投影到WGS84
# =========================
print("投影到WGS84...")
wgs84 = arcpy.SpatialReference(4326)
arcpy.management.Project(points_joined, out_grid_points_wgs84, wgs84)

# =========================
# 8) 添加 Province 字段并用 NAME 填充
# =========================
print("添加 Province 字段并用 NAME 填充...")
fields_out = [f.name for f in arcpy.ListFields(out_grid_points_wgs84)]
if "Province" not in fields_out:
    arcpy.management.AddField(out_grid_points_wgs84, "Province", "TEXT", field_length=100)

# 计算 Province = NAME
# 注：如果 join 后 NAME 变成 NAME_1，这里做个兼容处理
fields_out = [f.name for f in arcpy.ListFields(out_grid_points_wgs84)]
name_field = "NAME" if "NAME" in fields_out else ("NAME_1" if "NAME_1" in fields_out else None)
if name_field is None:
    raise RuntimeError("输出结果中找不到 NAME 或 NAME_1 字段，请检查 Spatial Join 输出字段。")

with arcpy.da.UpdateCursor(out_grid_points_wgs84, [name_field, "Province"]) as cursor:
    for n, p in cursor:
        cursor.updateRow((n, n))

# =========================
# 9) 添加经纬度字段
# =========================
print("添加经纬度字段...")
fields_out = [f.name for f in arcpy.ListFields(out_grid_points_wgs84)]
if "Longitude" not in fields_out:
    arcpy.management.AddField(out_grid_points_wgs84, "Longitude", "DOUBLE")
if "Latitude" not in fields_out:
    arcpy.management.AddField(out_grid_points_wgs84, "Latitude", "DOUBLE")

with arcpy.da.UpdateCursor(out_grid_points_wgs84, ["SHAPE@X", "SHAPE@Y", "Longitude", "Latitude"]) as cursor:
    for x, y, lon, lat in cursor:
        cursor.updateRow((x, y, x, y))

# =========================
# 10) 加载到当前地图（可选）
# =========================
print("添加结果到当前地图...")
project = arcpy.mp.ArcGISProject("CURRENT")
active_map = project.activeMap
active_map.addDataFromPath(out_grid_points_wgs84)

count = int(arcpy.management.GetCount(out_grid_points_wgs84)[0])
print(f"完成！中国范围内生成10km均匀网格点：{count} 个")
print("字段包含：Longitude, Latitude, ADCODE99, Province（由NAME复制而来）")
print(f"输出要素：{out_grid_points_wgs84}")
